In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import random
import pandas as pd

In [ ]:
def outlier_removal(df, column):

    df[column] = df[column].replace(-1, np.nan)

    r = df[column].dropna().to_numpy()
    
    if r.size == 0:
        print("Coluna não contém valores suficientes para análise.")
        return df

    r_max = np.max(r) 
    r = r / r_max  

    perc_min = []
    p_min = np.linspace(0.1, 2, 20)
    for i in p_min:
        perc_min.append(np.percentile(r, i))
    diff_perc_min = np.diff(perc_min)
    index_min = np.argmax(diff_perc_min)  
    thres_min = np.mean(perc_min[index_min:index_min + 2])

    perc_max = []
    p_max = np.linspace(98, 100, 20)
    for i in p_max:
        perc_max.append(np.percentile(r, i))
    diff_perc_max = np.diff(perc_max)
    index_max = np.argmax(diff_perc_max)  
    thres_max = np.mean(perc_max[index_max:index_max + 2])

    r_filtered = np.where((r < thres_min) | (r > thres_max), np.nan, r)

    r_filtered = r_filtered * r_max  

    df_filtered = df.copy()
    df_filtered.loc[~df[column].isna(), column] = r_filtered

    return df_filtered

def generate_missing_data(df, porcentagem):
    df_copy = df.copy()
    df_copy = outlier_removal(df, column='Throughput') # Removing outliers
    quantidade = (porcentagem * len(df_copy) / 100)
    indices_substituir = random.sample(df_copy.index.tolist(), round(quantidade))
    df_copy.loc[indices_substituir, 'Throughput'] = np.nan
    return df_copy

def rolling_imputation_analysis(df, original_df, missing_percentage, window_size=3, center=False, loop=False, method='median'):     
    # Ensure the 'Timestamp' column is in datetime format and set it as the index
    if 'Timestamp' in df.columns:
        df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
        original_df['Timestamp'] = pd.to_datetime(original_df['Timestamp'], errors='coerce')
        
        df = df.set_index('Timestamp')
        original_df = original_df.set_index('Timestamp')

    # Check for any remaining issues with the index format
    if not isinstance(df.index, pd.DatetimeIndex):
        print("The index is not a DatetimeIndex. Please ensure the 'Timestamp' column is in proper datetime format.")
        return df  # Return the original DataFrame if the conversion fails
    
    df = df.sort_index()
    original_df = original_df.sort_index()

    # Identify missing indices
    missing_indices = df[df['Throughput'].isnull()].index

    # Apply interpolation
    # df_imputed = df.interpolate(method=method, order=order, limit_direction=limit_direction)

    if (method == 'median'):
        if (loop):
            while df['Throughput'].isna().any():
                df['Throughput'] = df['Throughput'].fillna(df['Throughput'].rolling(window=window_size, center=center, min_periods=1).median())
        else:
            df_imputed = df['Throughput'] = df['Throughput'].fillna(df['Throughput'].rolling(window=window_size, center=center, min_periods=1).median())

    elif (method == 'average' or method == 'mean'):
        if (loop):
            while df['Throughput'].isna().any():
                df['Throughput'] = df['Throughput'].fillna(df['Throughput'].rolling(window=window_size, center=center, min_periods=1).mean())
        else:
            df_imputed = df['Throughput'] = df['Throughput'].fillna(df['Throughput'].rolling(window=window_size, center=center, min_periods=1).mean())

    elif (method != 'average' and method != 'mean' and method != 'median'):
        print ('Invalid roolling method')
        return
    
    print(df_imputed['Throughput'])
    # Plot the interpolated data
    # df_imputed['Throughput'].plot(style='.-', figsize=(12, 8), title=f'Throughput with Rolling {method}')
    # plt.scatter(missing_indices, df_imputed.loc[missing_indices, 'Throughput'], color='red')
    # print(df_imputed, missing_indices)
    # # Set plot labels
    # plt.xlabel('Time')
    # plt.ylabel('Throughput')
    # plt.show()

    # # Calculate RMSE for imputed values only
    # rmse = (np.sqrt(((original_df['Throughput'] - df_imputed['Throughput']).dropna() ** 2).mean()) / 1000000)

    # print(rmse)
    # # print(f"RMSE for imputed values: {rmse}")

    # result = {
    #     "Missing Percentage": missing_percentage,
    #     "Rolling Method": method,
    #     "Window Size": window_size,
    #     "Is on loop?": loop,
    #     "Center": center,
    #     "RMSE": rmse
    # }
    # return result, df_imputed

def plot_rmses_result(results_list):
    # Convert data into a DataFrame for easy manipulation
    df = pd.DataFrame(results_list)

    # Sort and extract unique values
    missing_percentages = sorted(df['Missing Percentage'].unique())
    methods = df['Interpolation Method'].unique()
    num_methods = len(methods)

    # Set up the figure
    fig, ax = plt.subplots(figsize=(12, 8))

    # X-axis positions for each group
    x = np.arange(len(missing_percentages))
    width = 0.15  # Width of each bar

    # Plot each interpolation method as a separate bar in each group
    for i, method in enumerate(methods):
        # Filter the DataFrame for the current interpolation method
        df_method = df[df['Interpolation Method'] == method]
        
        # RMSE values for the current method across all missing percentages
        rmse_values = [df_method[df_method['Missing Percentage'] == pct]['RMSE'].values[0] for pct in missing_percentages]
        
        # Plot the bars
        ax.bar(x + i * width, rmse_values, width, label=method)

    # Labeling and formatting
    ax.set_xlabel('Missing Percentage')
    ax.set_ylabel('RMSE')
    ax.set_title('RMSE by Missing Percentage and Interpolation Method')
    ax.set_xticks(x + width * (num_methods - 1) / 2)
    ax.set_xticklabels([f'{pct}%' for pct in missing_percentages])
    ax.legend(title='Interpolation Method')

    plt.show()

In [ ]:
#Using the longest interval among 07-07-2023 datasets
df = pd.read_csv("../datasets/throughput/07-07-2024/longest interval/treated cubic esmond data ap-rs 07-03-2023_longest_interval.csv")

In [ ]:
df_missing10 = generate_missing_data(df, 10)
df_missing20 = generate_missing_data(df, 20)
df_missing30 = generate_missing_data(df, 30)

In [ ]:
dfs_missing = [df_missing10, df_missing20, df_missing30]
rolling_results = []

In [ ]:
missing_percentage = 10
for missing in dfs_missing:
    result, _ = rolling_imputation_analysis(missing, df, missing_percentage, center=False, loop=False, method='mean')
    rolling_results.append(result)

    result2, _ = rolling_imputation_analysis(missing, df, missing_percentage, center=True, loop=False, method='mean')
    rolling_results.append(result2)

    result3, _ = rolling_imputation_analysis(missing, df, missing_percentage, center=True, loop=True, method='mean')
    rolling_results.append(result3)

    result4, _ = rolling_imputation_analysis(missing, df, missing_percentage, center=False, loop=True, method='mean')
    rolling_results.append(result4)


    result5, _ = rolling_imputation_analysis(missing, df, missing_percentage, center=False, loop=False, method='median')
    rolling_results.append(result5)

    result6, _ = rolling_imputation_analysis(missing, df, missing_percentage, center=True, loop=False, method='median')
    rolling_results.append(result6)

    result7, _ = rolling_imputation_analysis(missing, df, missing_percentage, center=True, loop=True, method='median')
    rolling_results.append(result7)

    result8, _ = rolling_imputation_analysis(missing, df, missing_percentage, center=False, loop=True, method='median')
    rolling_results.append(result8)

    missing_percentage = missing_percentage + 10

In [ ]:
plot_rmses_result(interpolation_results)